In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric as pyg

import torch_geometric.nn as pyg_nn
import torch_geometric.utils as pyg_utils

import time
from datetime import datetime

import networkx as nx
import numpy as np
import torch
import torch.optim as optim

from torch_geometric.data import  Data
from torch_geometric.loader import DataLoader

import torch_geometric.transforms as T

from tensorboardX import SummaryWriter
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from torch_geometric.utils import to_networkx

import matplotlib.font_manager as fm
from matplotlib import font_manager
import matplotlib.ticker as ticker

import scienceplots 


In [ ]:
# Use LaTeX-style fonts for professional look
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 24,  # Adjust based on target journal/conference
    "axes.labelsize": 26,
    "axes.titlesize": 24,
    "legend.fontsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "lines.linewidth": 1.5,  # Thicker lines for visibility
    "lines.markersize": 6,
    "grid.linestyle": "--",  # Dashed grid for subtlety
    "grid.alpha": 0.5,  # Slight transparency for readability
    "legend.frameon": False,  # No box around legends
    "figure.dpi": 300,  # High resolution
    "savefig.dpi": 300,  # High-resolution output
    "text.usetex": False,  # Use LaTeX for better typography (if available)
    "axes.grid": True,  # Enable grid
    "axes.spines.top": True,  # Hide top spine
    "axes.spines.right": True,  # Hide right spine
})

### Loading strain and stiffness reduction data

In [ ]:
stiffness_data_path = 'Data/Stiffness_Reduction'
strain_data_path = 'Data/Strain'

# Stiffness Data
stiff_file_paths = [f.path for f in os.scandir(stiffness_data_path) if f.path.endswith('.h5')]
stiff_file_paths.sort()
stiffness_dfs = {}
for i, file_path in enumerate(stiff_file_paths):
    stiffness_dfs[f'df{i}'] = pd.read_hdf(file_path)['Stiffness']

# Strain Data
strain_file_paths = [f.path for f in os.scandir(strain_data_path) if f.path.endswith('.h5')]
strain_file_paths.sort()
strain_dfs = {}
for i, file_path in enumerate(strain_file_paths):
    strain_dfs[f'df{i}'] = pd.read_hdf(file_path)



In [ ]:
def resample_stiffness_to_match_strain(strain_df, stiffness_df):
    strain_length = len(strain_df)
    stiffness_length = len(stiffness_df)
    
    # Assuming strain_df has a column with the strain values (e.g., 'strain')
    # and stiffness_df has the corresponding stiffness values in one or more columns.

    if strain_length > stiffness_length:
        # Interpolation: Upsample stiffness_df to match strain_df length
        # Assuming both dataframes are 1D for now, you can extend this to multiple columns later.
        x_old = np.linspace(0, 1, stiffness_length)  # Normalized index for stiffness
        x_new = np.linspace(0, 1, strain_length)  # Normalized index for strain

        # Interpolating f stiffness_df
        stiffness_df_resampled = pd.DataFrame(np.interp(x_new, x_old, stiffness_df))
    
    elif strain_length < stiffness_length:
        # Downsampling: Downsample stiffness_df to match strain_df length
        x_old = np.linspace(0, 1, stiffness_length)  # Normalized index for stiffness
        x_new = np.linspace(0, 1, strain_length)  # Normalized index for strain

        # Find the closest indices in stiffness_df to the new sample points
        idx_new = np.searchsorted(x_old, x_new)
        idx_new = np.clip(idx_new, 0, stiffness_length - 1)  # Ensure indices are valid

        stiffness_df_resampled = stiffness_df.iloc[idx_new].reset_index(drop=True)
    
    else:
        # If already the same length, no action required
        stiffness_df_resampled = stiffness_df.reset_index(drop=True)
    
    return stiffness_df_resampled


# def percentage_change_from_max(stiffness_df):
#     if isinstance(stiffness_df, pd.Series):
#         max_value = stiffness_df.max()
#         percentage_change_df = (stiffness_df / max_value) * 100
#         return percentage_change_df
#     elif isinstance(stiffness_df, pd.DataFrame):
#         max_value = stiffness_df.max().max()
#         percentage_change_df = (stiffness_df / max_value) * 100
#         return percentage_change_df
#     else:
#         raise ValueError("Input must be a pandas DataFrame or Series")
    

########## Correcting starting stiffness values ##########
def percentage_change_from_max(stiffness_df):
    if isinstance(stiffness_df, pd.Series):
        max_index = stiffness_df.idxmax()
        max_value = stiffness_df[max_index]
        percentage_change_df = (stiffness_df / max_value) * 100
        percentage_change_df.loc[:max_index] = 100  # Ensure correct assignment
        return percentage_change_df

    elif isinstance(stiffness_df, pd.DataFrame):
        percentage_change_df = stiffness_df.copy()
        max_values = stiffness_df.max()  # Get max for each column
        
        for col in stiffness_df.columns:
            max_idx_col = stiffness_df[col].idxmax()
            percentage_change_df[col] = (stiffness_df[col] / max_values[col]) * 100  # Normalize per column
            percentage_change_df.loc[:max_idx_col, col] = 100  # Set values before max to 100
        
        return percentage_change_df
    else:
        raise ValueError("Input must be a pandas DataFrame or Series")

In [ ]:
last_cycle = {}

for key in stiffness_dfs.keys():
    last_cycle[key] = len(stiffness_dfs[key])

last_cycle

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 9))

# Plot strain data with numerical index on primary x-axis
ax1.plot(range(len(strain_dfs['df4'])), strain_dfs['df4'])
ax1.set_xlabel('Index')
ax1.set_ylabel('Strain (με)')
ax1.tick_params(axis='both', which='major')

# Minor ticks & minor grid (sub-grid)
ax1.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))  # 4 sub-divisions per major tick
ax1.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))

# Create a secondary x-axis to display total seconds
ax2 = ax1.twiny()
ax2.set_xlim(ax1.get_xlim())

# Get the auto-generated ticks
ticks = ax1.get_xticks()
# Filter ticks to ensure they're within the valid range
valid_ticks = ticks[(ticks >= 0) & (ticks < len(strain_dfs['df4']))]
ax2.set_xticks(valid_ticks)

# Convert valid tick positions to total seconds and format as labels
valid_indices = valid_ticks.astype(int)
time_labels = (strain_dfs['df4'].index[valid_indices].total_seconds()).astype(int)
ax2.set_xticklabels(time_labels)
ax2.set_xlabel('Total Seconds (s)', labelpad=10)  # Add labelpad here
ax2.tick_params(axis='x', which='major')

ax2.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))

plt.tight_layout()
plt.show()


In [ ]:
#### Resample Strain, Smooth Strain and Stiffness Daata, and Match the Time Stamps ####

stiffness_post = {}
strain_post = {}
target_indexes = {}
stiffness_to_find_index = {}
# Use the key from strain_dfs
for key, strain_df in strain_dfs.items():
    
    if key == 'df2':
        strain_df = strain_df.iloc[:,:-8]
        
    # Resample the strain data and smooth it with rolling mean
    strain_resampled = strain_df.resample("200s").mean().rolling(10).mean()

    # Drop the NaN values
    strain_resampled = strain_resampled.dropna()
    strain_post[key] = strain_resampled

    print(strain_post[key].shape)

# Plot of the preprocessed strain data for FOD7
fig, ax1 = plt.subplots(figsize=(16, 9))
ax1.plot(range(len(strain_post['df4'])), strain_post['df4'])
ax1.set_xlabel('Time (x200s)')
ax1.set_ylabel('Strain (με) ')
ax1.tick_params(axis='both', which='major')

# Minor ticks & minor grid (sub-grid)
ax1.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))  # 4 sub-divisions per major tick
ax1.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))


# Create a secondary x-axis to display cycles
ax2 = ax1.twiny()

# Generate cycle numbers
cycle_numbers = np.linspace(0, last_cycle['df4'], len(strain_post['df4']))

# Set the limits for the secondary x-axis
ax2.set_xlim(ax1.get_xlim())

# Get the current tick locations from the primary x-axis
primary_ticks = ax1.get_xticks()

# Filter out any ticks that are outside the data range
primary_ticks = primary_ticks[(primary_ticks >= 0) & (primary_ticks <= len(strain_post['df4']))]

# Convert the primary tick locations to cycle numbers
cycle_tick_labels = np.interp(primary_ticks, np.linspace(0, len(strain_post['df4']), len(cycle_numbers)), cycle_numbers)

# Set the tick locations and labels for the secondary x-axis
ax2.set_xticks(primary_ticks)
ax2.set_xticklabels(cycle_tick_labels.astype(int))

# Set the label for the secondary x-axis
ax2.set_xlabel('Cycle Number', labelpad=10)
ax2.tick_params(axis='x', which='major')

ax2.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))

plt.tight_layout()
plt.show()


In [ ]:
#### Resample Strain, Smooth Strain and Stiffness Daata, and Match the Time Stamps ####

stiffness_post = {}
strain_post = {}
target_indexes = {}
stiffness_to_find_index = {}
# Use the key from strain_dfs
for key, strain_df in strain_dfs.items():
    
    if key == 'df2':
        strain_df = strain_df.iloc[:,:-8]
        
    # Resample the strain data and smooth it with rolling mean
    strain_resampled = strain_df.resample("200s").mean().rolling(10).mean()

    # Drop the NaN values
    strain_resampled = strain_resampled.dropna()


    ##### Custom Feature Engineering #####
    ################################################
    strain_temp= np.cumsum(abs(np.diff(strain_resampled, axis=0)), axis=0)
    strain_temp= pd.DataFrame(strain_temp)
    # copy index from strain_resampled
    strain_temp.index = strain_resampled.iloc[1:,:].index
    strain_resampled = strain_temp
    #strain_resampled = strain_resampled.dropna()
    #####################################################
    
    # Store the resampled strain in strain_post
    strain_post[key] = strain_resampled
    
    # Get the corresponding stiffness_df using the same key from stiffness_dfs
    stiffness_df = stiffness_dfs[key].rolling(50).mean()

    # Drop the NaN values
    stiffness_df = stiffness_df.dropna()

    # Calculate the percentage change from the maximum value
    stiffness_df = percentage_change_from_max(stiffness_df)
    
    # Resample the stiffness data to match the strain
    stiffness_resampled = resample_stiffness_to_match_strain(strain_resampled, stiffness_df)

    stiffness_to_find_index[key] = stiffness_resampled.copy()

    
    # Store the resampled stiffness in stiffness_post
    stiffness_post[key] = pd.DataFrame(stiffness_resampled)

    stiffness_post[key].index = strain_post[key].index

    # #### RUL ESTIMATION ###########
    stiffness_post[key] = pd.DataFrame(np.linspace(stiffness_post[key].index.total_seconds()[-1], 0, len(stiffness_post[key])))

    
    # # #########################################
    
    stiffness_post[key].index = strain_post[key].index

    # # print the NaN values in the stiffness and strain data
    # print(f"NaN values in {key}:")
    # print(f"Stiffness: {stiffness_post[key].isna().sum().sum()}")
    # print(f"Strain: {strain_post[key].isna().sum().sum()}")

In [ ]:
# Plot together strain and stiffness with rescaled x-axis
strain_x_rescaled = {}
stiffness_x_rescaled = {}


fig, ax = plt.subplots(figsize=(16,9))
# Retrieve original x values
strain_x = strain_post["df4"].index.total_seconds()
stiffness_x = stiffness_post["df4"].index.total_seconds()

# Scale the x-axis so the last point corresponds to last_cycle[key]
max_time = max(strain_x.max(), stiffness_x.max())
strain_x_rescaled["df4"] = strain_x * (last_cycle["df4"] / max_time)
stiffness_x_rescaled["df4"] = stiffness_x * (last_cycle["df4"] / max_time)

# Plot strain data
for column in strain_post["df4"].columns:
    ax.scatter(strain_x_rescaled["df4"], strain_post["df4"][column].values, label=column, s=1)

# Plot stiffness data
#ax.scatter(stiffness_x_rescaled["df4"], stiffness_post["df4"], label='RUL', s=1)

# Customize legend and axes
#ax.legend(loc='best', fontsize='x-small', ncol=2)
ax.set_title("Health Indicator curves FOD7  ")
ax.set_xlabel(f"Cycles")
ax.set_ylabel("HI")
ax.tick_params(axis='both', which='major')

plt.show()

In [ ]:
# RUL Plot
fig, ax = plt.subplots(figsize=(16,9))

strain_x_rescaled = {}
stiffness_x_rescaled = {}
num = 2

for key in stiffness_post.keys():
    
    # Retrieve original x values
    strain_x = strain_post[key].index.total_seconds()
    stiffness_x = stiffness_post[key].index.total_seconds()

    # Scale the x-axis so the last point corresponds to last_cycle[key]
    max_time = max(strain_x.max(), stiffness_x.max())
    strain_x_rescaled[key] = strain_x * (last_cycle[key] / max_time)
    stiffness_x_rescaled[key] = stiffness_x * (last_cycle[key] / max_time)
    stiffness_post[key] = stiffness_post[key] * (last_cycle[key] / max_time)

    # RUL adjustment

    
    num = num + 1
    # Plot stiffness data
    ax.scatter(stiffness_x_rescaled[key], stiffness_post[key], label=f'FOD{num}', s=1)

    # Customize legend and axes
    ax.legend(loc='best', ncol=2, markerscale=3, fontsize='x-small')
    ax.set_title(f"Stiffness reduction per cycles")
    ax.set_xlabel(f"Cycles")
    ax.set_ylabel("Stiffness")
    ax.tick_params(axis='both', which='major')

plt.show()

In [ ]:
### Normalize the data
min_stiffness = {}
max_stiffness = {}

for key, _ in strain_post.items():
    
    # normalize both strain and stigffness to [0,1]
    strain_post[key] = (strain_post[key] - strain_post[key].min()) / (strain_post[key].max() - strain_post[key].min())
    min_stiffness[key] = stiffness_post[key].values.min()
    max_stiffness[key] = stiffness_post[key].values.max()
    stiffness_post[key] = (stiffness_post[key] - min_stiffness[key]) / (max_stiffness[key] - min_stiffness[key]) # normalize for training
    stiffness_to_find_index[key] = (stiffness_to_find_index[key] - stiffness_to_find_index[key].min())/ (stiffness_to_find_index[key].max() - stiffness_to_find_index[key].min()) # normalize for training
    
    #stiffness_post[key] = (stiffness_post[key] / stiffness_post[key].max())  # Normalize to show the percentage of stiffness change
    # store the min and max values for each key calculated from the stiffness data


In [ ]:
target_indexes = {}

def find_closest_index(array, target):
    # Find index of the closest value to the target in the array
    return np.abs(array - target).argmin()

for key, values in stiffness_to_find_index.items():
    # Convert the list of stiffness values to a NumPy array for efficient operations
    stiffness_values = np.array(values)
    
    # Find the closest index of the value 100
    closest_index_99 = find_closest_index(stiffness_values, 0.99)
    value_100 = stiffness_values[closest_index_99]
    

    # Initialize variables
    index_99 = None
    valid_95 = []
    valid_90 = []
    valid_85 = []

  
    index_99 = closest_index_99
    # Filter out values before the index_99
    filtered_values = stiffness_values[index_99 + 1:]
   
    
# Find the closest index in the filtered array
    target_indexes[key] = {
        0.99: index_99,
        0.95: find_closest_index(filtered_values, 0.95) + index_99 + 1,
        0.90: find_closest_index(filtered_values, 0.90) + index_99 + 1,
        0.85: find_closest_index(filtered_values, 0.85) + index_99 + 1,
        None: len(stiffness_values) - 1
    }

In [ ]:
target_indexes

In [ ]:
# PLot the RUL with the stiffness threshold markers



# import matplotlib.lines as mlines

# # Initialize lists to store legend handles and labels
# fod_handles = []
# fod_labels = []
# threshold_handles = []
# threshold_labels_list = []

# # RUL Plot with Stiffness Threshold Markers
# fig, ax = plt.subplots(figsize=(16, 9))
# num = 2

# # Define markers for thresholds - using only black and white
# threshold_markers = {0.99: 'o', 0.95: 's', 0.90: '^', 0.85: 'd', None: 'x'}
# threshold_colors = {0.99: 'white', 0.95: 'white', 0.90: 'white', 0.85: 'white', None: 'black'}
# threshold_labels = {0.99: '99% Stiffness', 0.95: '95% Stiffness', 
#                     0.90: '90% Stiffness', 0.85: '85% Stiffness', None: 'Failure Point'}

# # Track if we've added the threshold to the legend already
# threshold_in_legend = {k: False for k in threshold_markers.keys()}

# for key in stiffness_post.keys():
    
#     num = num + 1
#     panel_name = f'FOD{num}'

#     # unnormalize the stiffness_post
#     stiffness_post[key] = (stiffness_post[key] * (max_stiffness[key] - min_stiffness[key])) + min_stiffness[key]
    
#     # Plot stiffness data
#     line = ax.scatter(stiffness_x_rescaled[key], stiffness_post[key], label=panel_name, s=1)
#     fod_handles.append(line)
#     fod_labels.append(panel_name)
    
#     # Add threshold markers
#     for threshold, idx in target_indexes[key].items():
#         if idx >= len(stiffness_x_rescaled[key]):
#             # Skip if index is out of bounds
#             continue
            
#         x_val = stiffness_x_rescaled[key][idx]
#         y_val = float(stiffness_post[key].iloc[idx])
        
#         # Determine whether to include in legend
#         if not threshold_in_legend[threshold]:
#             label = threshold_labels[threshold]
#             threshold_in_legend[threshold] = True
#         else:
#             label = None
            
#         # Plot the marker - with appropriate handling for filled vs unfilled markers
#         if threshold is None:
#             # For 'x' marker which doesn't support edgecolors
#             marker = ax.scatter(x_val, y_val, 
#                                 marker=threshold_markers[threshold], 
#                                 color='black',
#                                 s=50, 
#                                 label=label,
#                                 zorder=10)
#         else:
#             # For filled markers - use white fill with black edge
#             marker = ax.scatter(x_val, y_val, 
#                                 marker=threshold_markers[threshold], 
#                                 color='white',
#                                 s=50, 
#                                 label=label,
#                                 edgecolors='black',
#                                 linewidth=1.5,
#                                 zorder=10)
        
#         if label:
#             threshold_handles.append(marker)
#             threshold_labels_list.append(threshold_labels[threshold])

# # Create custom legend handles for FOD labels (dots with a specific size)
# fod_handles_custom = []
# for handle in fod_handles:
#     fod_handles_custom.append(mlines.Line2D([], [], marker='o', color='w', markerfacecolor=handle.get_facecolor(),
#                                             markersize=8, label=handle.get_label()))  # Adjust markersize as needed



# # Customize legend with handles and labels
# handles = fod_handles_custom + threshold_handles
# labels = fod_labels + threshold_labels_list
# ax.legend(handles=handles, labels=labels, loc='best', ncol=2, fontsize='small')

# # Customize plot axes
# ax.set_title("RUL Histories with Stiffness Degradation Thresholds")
# ax.set_xlabel("Cycles")
# ax.set_ylabel("RUL (cycles)")
# ax.tick_params(axis='both', which='major')
# ax.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()


In [ ]:
# plot together strain and stiffness


for key, _ in strain_post.items():
    plt.figure(figsize=(16, 9))
    for column in strain_post[key].columns:
        plt.scatter(strain_post[key].index.total_seconds(), strain_post[key][column].values, label=column, s=1)
    #plt.scatter(stiffness_post[key].index.total_seconds(), stiffness_post[key], label='Stiffness', s=1)
    # control legend size and skip the middle labels

    plt.title("Input Data - Target Data")
    plt.xlabel("Time (s)")
    plt.ylabel("Normalized Values (strain, stiffness)")
    plt.show()



In [ ]:
# Plot together strain and stiffness with rescaled x-axis
strain_x_rescaled = {}
stiffness_x_rescaled = {}

for key, _ in strain_post.items():
    plt.figure(figsize=(16, 9))
    # Retrieve original x values
    strain_x = strain_post[key].index.total_seconds()
    stiffness_x = stiffness_post[key].index.total_seconds()
    
    # Scale the x-axis so the last point corresponds to last_cycle[key]
    max_time = max(strain_x.max(), stiffness_x.max())
    strain_x_rescaled[key] = strain_x * (last_cycle[key] / max_time)
    stiffness_x_rescaled[key] = stiffness_x * (last_cycle[key] / max_time)

    # Plot strain data
    for column in strain_post[key].columns:
        plt.scatter(strain_x_rescaled[key], strain_post[key][column].values, label=column, s=1)
    
    # Plot stiffness data
    plt.scatter(stiffness_x_rescaled[key], stiffness_post[key], label='RUL', s=1)
    
    # Customize legend and axes
    plt.legend(loc='best', fontsize='x-small', ncol=2)
    plt.title(f"Input Data - Target Data (FOD{int(key.split('f')[-1])+3})")
    plt.xlabel(f"Cycles")
    plt.ylabel("Normalized Values (strain, RUL)")
    
    plt.show()

In [ ]:
# Cut the data to the target indexes based on the stiffness drop percentage
drop = 0.99 # None, 0.85, 0.90, 0.95, 1.0  choose the drop percentage

In [ ]:
##################################################################
stiffness_post = {}
strain_post = {}
stiffness_to_find_index = {}

# Use the key from strain_dfs
for key, strain_df in strain_dfs.items():
    
    if key == 'df2':
        strain_df = strain_df.iloc[:,:-8]
    
    # Resample the strain data and smooth it with rolling mean
    strain_resampled = strain_df.resample("200s").mean().rolling(10).mean()
    # Drop the NaN values
    strain_resampled = strain_resampled.dropna()
    ##### Custom Feature Engineering #####
    ################################################
    strain_temp= np.cumsum(abs(np.diff(strain_resampled, axis=0)), axis=0)
    strain_temp= pd.DataFrame(strain_temp)
    # copy index from strain_resampled
    strain_temp.index = strain_resampled.iloc[1:,:].index
    strain_resampled = strain_temp
    #strain_resampled = strain_resampled.dropna()
    #####################################################
    # Store the resampled strain in strain_post
    strain_post[key] = strain_resampled
    # Get the corresponding stiffness_df using the same key from stiffness_dfs
    stiffness_df = stiffness_dfs[key].rolling(50).mean()
    # Drop the NaN values
    stiffness_df = stiffness_df.dropna()
    # Calculate the percentage change from the maximum value
    stiffness_df = percentage_change_from_max(stiffness_df)
    # Resample the stiffness data to match the strain
    stiffness_resampled = resample_stiffness_to_match_strain(strain_resampled, stiffness_df)
    stiffness_to_find_index[key] = stiffness_resampled.copy()
    # Store the resampled stiffness in stiffness_post
    stiffness_post[key] = pd.DataFrame(stiffness_resampled)
    stiffness_post[key].index = strain_post[key].index
    # #### RUL ESTIMATION ###########
    stiffness_post[key] = pd.DataFrame(np.linspace(stiffness_post[key].index.total_seconds()[-1], 0, len(stiffness_post[key])))
    # # #########################################
    stiffness_post[key].index = strain_post[key].index
    cut_index = target_indexes[key][drop]
    stiffness_post[key] = stiffness_post[key].iloc[:cut_index+1]
    strain_post[key] = strain_post[key].iloc[:cut_index+1]
    ### Normalize the data
min_stiffness = {}
max_stiffness = {}

for key, _ in strain_post.items():
    
    # normalize both strain and stigffness to [0,1]
    strain_post[key] = (strain_post[key] - strain_post[key].min()) / (strain_post[key].max() - strain_post[key].min())
    min_stiffness[key] = stiffness_post[key].values.min()
    max_stiffness[key] = stiffness_post[key].values.max()
    stiffness_post[key] = (stiffness_post[key] - min_stiffness[key]) / (max_stiffness[key] - min_stiffness[key]) # normalize for training
    stiffness_to_find_index[key] = (stiffness_to_find_index[key] - stiffness_to_find_index[key].min())/ (stiffness_to_find_index[key].max() - stiffness_to_find_index[key].min()) # normalize for training
    
    #stiffness_post[key] = (stiffness_post[key] / stiffness_post[key].max())  # Normalize to show the percentage of stiffness change
    # store the min and max values for each key calculated from the stiffness data


In [ ]:
# plot together strain and stiffness
# Plot together strain and stiffness with rescaled x-axis

for key, _ in strain_post.items():
    plt.figure(figsize=(16, 9))
    # Retrieve original x values
    cut_index = target_indexes[key][drop]
    strain_x_rescaled[key] = strain_x_rescaled[key][:cut_index+1] 
    stiffness_x_rescaled[key] = stiffness_x_rescaled[key][:cut_index+1]

    # Plot strain data
    for column in strain_post[key].columns:
        plt.scatter(strain_x_rescaled[key], strain_post[key][column].values, label=column, s=1)
    
    # Plot stiffness data
    plt.scatter(stiffness_x_rescaled[key], stiffness_post[key], label='Stiffness', s=1)
    
    # Customize legend and axes
    plt.legend(loc='best', fontsize='x-small', ncol=2)
    plt.title(f"Input Data - Target Data (FOD{int(key.split('f')[-1])+3})")
    plt.xlabel(f"Cycles")
    plt.ylabel("Normalized Values (strain, RUL)")
    
    plt.show()


In [ ]:
# Leave out FOD3 sinnce it has only 6 sensors

strain_data = []
stiffness_data = []

for key, _ in strain_post.items():
    if key == 'df0':
        continue
    strain_data.append(strain_post[key].values)
    
    # Reshape stiffness data to ensure it is (N, 1)
    stiffness_values = stiffness_post[key].values
    if stiffness_values.ndim == 1:
        stiffness_values = stiffness_values.reshape(-1, 1)
    
    stiffness_data.append(stiffness_values)


In [ ]:
for strain , stiffness in zip(strain_data, stiffness_data):
    print(strain.shape, stiffness.shape)
    

### Constructing data structure for Graph NN

In [ ]:
specimen_data = []

# Assuming strain_data and stiffness_data are lists of tensors
for specimen in range(4):
    strain_data_specimen = torch.tensor(strain_data[specimen], dtype=torch.float)  # Ensure it is a tensor
    stiffness_specimen = torch.tensor(stiffness_data[specimen], dtype=torch.float)  # Ensure it is a tensor
    
    data_list = []  # List to store Data objects for each time step within a specimen
    
    for t in range(strain_data_specimen.shape[0]):  
        x = strain_data_specimen[t].reshape(-1, 1)           # Shape: (16, 1)
        
        # Create edge index for a fully connected graph (excluding self-loops)
        edge_index = []
        for i in range(16):
            for j in range(16):
                if i != j:
                    edge_index.append([i, j])
        
        # Convert edge_index to tensor and transpose it
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()  # Shape: (2, 240)
        
        # Create edge attributes (e.g., zero attributes)
        edge_attr = torch.zeros(edge_index.size(1), 1, dtype=torch.float)         # Shape: (240, 1)

        # Extract target value for the current time step
        y = stiffness_specimen[t].reshape(-1,1)  # Assuming single value target, reshape to (1, 1)

        # Create Data object for the current time step
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
        data_list.append(data)
    
    specimen_data.append(data_list)

In [ ]:
# def visualize_graph(data):
#     edge_index = data.edge_index
#     edge_attr = data.edge_attr
#     x = data.x
#     y = data.y
    
#     # Create a networkx graph
#     G = nx.Graph()
    
#     # Add nodes with strain attribute
#     for i in range(x.size(0)):
#         G.add_node(i, strain=x[i].item())
    
#     # Add edges with stiffness attribute
#     for i in range(edge_index.size(1)):
#         src, dst = edge_index[:, i].tolist()
#         G.add_edge(src, dst, stiffness=edge_attr[i].item())
    
#     # Layout for node positions
#     pos = nx.spring_layout(G)

#     # Node colors based on strain
#     node_colors = [G.nodes[node]['strain'] for node in G.nodes]
#     nodes_cmap = plt.cm.viridis  # Colormap for nodes
#     norm_nodes = mcolors.Normalize(vmin=min(node_colors), vmax=max(node_colors))
#     node_colors_normalized = [norm_nodes(value) for value in node_colors]
    
#     # Edge colors based on stiffness
#     edge_colors = [G.edges[edge]['stiffness'] for edge in G.edges]
#     edges_cmap = plt.cm.plasma  # Colormap for edges
#     norm_edges = mcolors.Normalize(vmin=min(edge_colors), vmax=max(edge_colors))
#     edge_colors_normalized = [norm_edges(value) for value in edge_colors]

#     # Create a figure with subplots to avoid the colorbar issue
#     fig, ax = plt.subplots(figsize=(8, 8))

#     # Draw the graph with node and edge coloring
#     nodes = nx.draw_networkx_nodes(
#         G, pos, node_color=node_colors_normalized, cmap=nodes_cmap, node_size=500, ax=ax
#     )
#     edges = nx.draw_networkx_edges(
#         G, pos, edge_color=edge_colors_normalized, edge_cmap=edges_cmap, width=2, ax=ax
#     )
#     nx.draw_networkx_labels(G, pos, ax=ax)

#     # Add color bars explicitly linked to the current figure and axes
#     sm_nodes = plt.cm.ScalarMappable(cmap=nodes_cmap, norm=norm_nodes)
#     sm_nodes.set_array(node_colors)
#     fig.colorbar(sm_nodes, ax=ax, label="Node Strain", shrink=0.7)
    
#     sm_edges = plt.cm.ScalarMappable(cmap=edges_cmap, norm=norm_edges)
#     sm_edges.set_array(edge_colors)
#     fig.colorbar(sm_edges, ax=ax, label="Edge Stiffness", shrink=0.7)
    
#     plt.title(f"Graph Visualization: Strain and Stiffness")
#     plt.show()

# # Visualize a sample graph
# visualize_graph(train_data[1000])

## Model

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_dim, output_dim, dropout):
        super(GNN, self).__init__()
        self.fc0 = torch.nn.Linear(num_node_features, hidden_dim)
        self.norm1 = torch.nn.LayerNorm(hidden_dim)  # Layer normalization after first linear layer
        
        self.conv1 = pyg_nn.GCNConv(hidden_dim, hidden_dim)
        self.norm2 = torch.nn.LayerNorm(hidden_dim)  # Layer normalization after first GCNConv
        
        self.conv2 = pyg_nn.GCNConv(hidden_dim, hidden_dim)
        self.norm3 = torch.nn.LayerNorm(hidden_dim)  # Layer normalization after second GCNConv
        
        self.fc1 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.norm4 = torch.nn.LayerNorm(hidden_dim)  # Layer normalization after first FC layer
        
        self.fc2 = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        # First linear layer + normalization + activation
        x = self.fc0(x)
        #x = self.norm1(x)
        x = F.leaky_relu(x)
        #x = self.dropout(x)
        
        # First GCN layer + normalization + activation
        x = self.conv1(x, edge_index)
        #x = self.norm2(x)
        x = F.leaky_relu(x)
        #x = self.dropout(x)
        
        # Second GCN layer + normalization + activation
        x = self.conv2(x, edge_index)
        emb = x  # Embedding after second GCNConv
        #x = self.norm3(x)
        x = F.leaky_relu(x)
        #x = self.dropout(x)
        
        # Global mean pooling
        x = pyg_nn.global_mean_pool(x, batch)
        
        # Fully connected layers + normalization + activation
        x = self.fc1(x)
        x = self.norm4(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        
        return emb, x

    def loss(self, pred, true):
        return F.mse_loss(pred, true)

In [ ]:
# print lernable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# build model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN(1, 15, 1, dropout=0.1).to(device)
opt = optim.Adam(model.parameters(), lr=0.01)
# Learning Rate Scheduler based on training loss
scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=5)
print(model)
print(f"Model has {count_parameters(model):,} trainable parameters.")
# print the available data points
temp_len = 0
for data in specimen_data:
    temp_len += len(data)
print(f'Total data points: {temp_len}')


## Training Loop

In [ ]:
def weighted_mse_loss(pred, target, weight_range=(0, 0.3), weight_value=2.0):
    """
    Weighted MSE loss giving higher importance to target values within a specific range.
    
    Parameters:
        pred (Tensor): Predicted values (batch_size, num_outputs).
        target (Tensor): Target values (batch_size, num_outputs).
        weight_range (tuple): Range of target values to apply the weight.
        weight_value (float): Weight factor to apply to the loss for targets within the range.
    
    Returns:
        Tensor: Weighted MSE loss.
    """
    # Calculate the standard MSE loss
    mse_loss = F.mse_loss(pred, target, reduction='none')  # Keep per-element loss
    
    # Create a weight mask
    weight_mask = (target >= weight_range[0]) & (target <= weight_range[1])
    weights = torch.ones_like(target)
    weights[weight_mask] = weight_value  # Apply higher weight to values in the range
    
    # Apply weights to the loss
    weighted_loss = mse_loss * weights
    return weighted_loss.mean()  # Reduce to a single loss value

In [ ]:
def train(train_loader, val_loader, writer, epochs, patience=10):
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        for data in train_loader:
            data = data.to(device)
            data.x = data.x.float()
            data.y = data.y.float()
            opt.zero_grad()
            emb, pred = model(data)
            # Use the weighted loss function
            loss = weighted_mse_loss(pred, data.y, weight_range=(0, 0.2), weight_value=6)
            #loss = model.loss(pred, data.y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * data.num_graphs

        total_loss /= len(train_loader.dataset)
        train_losses.append(total_loss)
        writer.add_scalar("loss", total_loss, epoch)

        val_mse = validation(val_loader, model)
        val_losses.append(val_mse)
        print(f"Epoch {epoch}. Loss: {total_loss:.4f}. Val MSE: {val_mse:.4f}")
        writer.add_scalar("val_mse", val_mse, epoch)

        scheduler.step(total_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch}. Current learning rate: {current_lr:.6f}")
        writer.add_scalar("learning_rate", current_lr, epoch)

        if epoch < 3:
            continue
        else:
            if val_mse < best_val_loss:
                best_val_loss = val_mse
                best_model_state = model.state_dict()
                torch.save(best_model_state, "best_model/best_model_state.pth")
                print(f"New best model found at epoch {epoch} with Val MSE: {val_mse:.4f}")
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                print(f"No improvement at epoch {epoch}. Patience: {epochs_without_improvement}/{patience}")

        if epochs_without_improvement >= patience:
            print(f"Early stopping after {epoch} epochs. Best Val MSE: {best_val_loss:.4f}")
            break

    if best_model_state is not None:
        model.load_state_dict(torch.load("best_model/best_model_state.pth", weights_only=True))
        print("Loaded the best model state.")
    else:
        print("No best model state was saved.")

    return model, train_losses, val_losses

def validation(val_loader, model):
    model.eval()
    total_loss = 0

    for data in val_loader:
        with torch.no_grad():
            data = data.to(device)
            emb, pred = model(data)
            loss = model.loss(pred, data.y)
            total_loss += loss.item() * data.num_graphs

    return total_loss / len(val_loader.dataset)

In [ ]:
# FOD 4 is df1 and specimen_data[0]
# FOD 5 is df2 and specimen_data[1]
# FOD 6 is df3 and specimen_data[2]
# FOD 7 is df4 and specimen_data[3]

test_key = 'df4'

# Split the specimens
train_data = specimen_data[0] +specimen_data[1]  + specimen_data[2]
val_data = specimen_data[3]                                           
test_data = specimen_data[3]                                         

# Create DataLoaders for each set
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)  # Shuffle during training
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)     # No shuffle for validation
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)   # No shuffle for testing

writer = SummaryWriter("./log/" + datetime.now().strftime("%Y%m%d-%H%M%S"))

model, train_losses, val_losses = train(train_loader, val_loader, writer,epochs=100,  patience=30)
with plt.style.context(['science', 'no-latex']):
    fig_width = 16
    fig_height = 9
    plt.figure(figsize=(fig_width, fig_height))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()


## Inference

In [ ]:
def inference(model, dataset, device, key):
    model.eval()
    
    # For storing all predictions and true values
    all_true_unnorm = []
    all_pred_unnorm = []
    
    for data in dataset:
        with torch.no_grad():
            data = data.to(device)
            emb, pred = model(data)
            
            # Normalized true and predicted values
            true_val_norm = data.y.cpu().numpy().flatten()
            pred_val_norm = pred.cpu().numpy().flatten()
            
            # Unnormalize
            true_val_unnorm = true_val_norm * strain_x_rescaled[key][-1]
            pred_val_unnorm = pred_val_norm * strain_x_rescaled[key][-1]
            
            # Collect all values
            all_true_unnorm.extend(true_val_unnorm)
            all_pred_unnorm.extend(pred_val_unnorm)
    
    # Convert to numpy arrays for calculations
    all_true_unnorm = np.array(all_true_unnorm)
    all_pred_unnorm = np.array(all_pred_unnorm)
    
    # Calculate metrics correctly on all data points
    mae = np.mean(np.abs(all_true_unnorm - all_pred_unnorm))
    rmse = np.sqrt(np.mean((all_true_unnorm - all_pred_unnorm) ** 2))
    
    # Calculate MAPE
    non_zero_indices = all_true_unnorm != 0
    mape = np.mean(np.abs((all_true_unnorm[non_zero_indices] - all_pred_unnorm[non_zero_indices]) / 
                          all_true_unnorm[non_zero_indices])) * 100
    
    return all_true_unnorm, all_pred_unnorm, mae, rmse, mape

def plot_predictions(true_values, predicted_values, mae, rmse, mape, title, key='df3'):
    # Define figure size for 16:9 aspect ratio
    fig_width = 16
    fig_height = 9
    plt.figure(figsize=(fig_width, fig_height))
    
    plt.plot(strain_x_rescaled[key], true_values, label="True Values", color="b")
    plt.plot(strain_x_rescaled[key], predicted_values, label="Predicted Values", color="r", linestyle='--')
    
    # Create metrics text with proper alignment using monospace font
    metrics_text = (
        f"MAE:   {mae:.2f} \n"
        f"RMSE:   {rmse:.2f} \n"
        f"MAPE:   {mape:.2f}%"
    )
    
    # Position the text box in the upper right corner with better styling
    plt.annotate(metrics_text, xy=(0.95, 0.95), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.6", facecolor='white', alpha=0.8, 
                           edgecolor='gray', linewidth=1),
                 ha='right', va='top', fontsize=14, family='monospace')
    
    plt.xlabel("Cycles")
    plt.ylabel("RUL (Cycles)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def enable_dropout(model):
    """ Function to enable the dropout layers during test-time """
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.train()

def mc_dropout_inference(model, dataset, device, num_samples=100, key='df3'):
    model.eval()
    enable_dropout(model)  # Enable dropout during inference
    
    true_values_unnormalized = []
    mc_predicted_values_unnormalized = []  # Store all the MC dropout predictions (unnormalized)
    total_mae_unnormalized = 0
    total_mse_unnormalized = 0  # Keep tracking MSE for RMSE calculation
    total_mape_unnormalized = 0
    count = 0
    
    for data in dataset:
        data = data.to(device)
        mc_preds = []
        
        for _ in range(num_samples):  # Perform `num_samples` stochastic forward passes
            with torch.no_grad():
                emb, pred = model(data)
                mc_pred = pred.cpu().numpy().flatten()
                mc_preds.append(mc_pred)  # Keep predictions normalized initially

        mc_preds = np.array(mc_preds)  # Convert to numpy array for easier calculations
        
        # Mean and standard deviation of the predictions (for confidence intervals)
        mean_pred = mc_preds.mean(axis=0)
        std_pred = mc_preds.std(axis=0)
        
        true_val_norm = data.y.cpu().numpy().flatten()

        # Unnormalize for unnormalized metric calculations
        true_val_unnorm = true_val_norm * strain_x_rescaled[key][-1]
        mean_pred_unnorm = mean_pred * strain_x_rescaled[key][-1]
        std_pred_unnorm = std_pred * strain_x_rescaled[key][-1]

        # Calculate MAE using unnormalized values
        mae_unnorm = np.mean(np.abs(true_val_unnorm - mean_pred_unnorm))
        total_mae_unnormalized += mae_unnorm
        
        # Calculate MSE for RMSE calculation
        mse_unnorm = np.mean((true_val_unnorm - mean_pred_unnorm) ** 2)
        total_mse_unnormalized += mse_unnorm

        # Calculate MAPE using unnormalized values, skipping zero true values
        for t, p in zip(true_val_unnorm, mean_pred_unnorm):
            if t != 0:
                total_mape_unnormalized += np.abs((t - p) / t) * 100
                count += 1

        # Store unnormalized values for plotting or further analysis
        true_values_unnormalized.append(true_val_unnorm)
        mc_predicted_values_unnormalized.append((mean_pred_unnorm, std_pred_unnorm))

    # Final metrics using unnormalized values
    mae = total_mae_unnormalized / len(dataset)
    mse = total_mse_unnormalized / len(dataset)  # Still calculate MSE for RMSE
    rmse = np.sqrt(mse)
    avg_mape = total_mape_unnormalized / count if count > 0 else float('inf')
    
    return true_values_unnormalized, mc_predicted_values_unnormalized, mae, rmse, avg_mape


def plot_mc_predictions(true_values, mc_predicted_values, mae, rmse, mape, title, target_indexes=None, confidence_level=1.96, key='df3'):
    """
    Plots the true values, predicted mean values, and confidence intervals.

    Args:
    - true_values: List or array of true values (unnormalized).
    - mc_predicted_values: List of tuples (mean, std) from MC dropout predictions (unnormalized).
    - mae: Mean Absolute Error of the predictions (calculated from normalized data).
    - rmse: Root Mean Squared Error of the predictions (calculated from normalized data).
    - mape: Mean Absolute Percentage Error of the predictions (calculated from normalized data).
    - title: Title for the plot.
    - target_indexes: Dictionary with target indexes for adding arrows (optional).
    - confidence_level: z-score for the desired confidence interval (default 1.96 for 95% CI).
    """
    # Extract the mean and std from the predicted values
    mean_pred = np.array([mean for mean, std in mc_predicted_values])
    std_pred = np.array([std for mean, std in mc_predicted_values])

    # Flatten the arrays to ensure they are 1D
    mean_pred = mean_pred.flatten()
    std_pred = std_pred.flatten()

    # Calculate upper and lower confidence bounds
    upper_bound = mean_pred + confidence_level * std_pred
    lower_bound = mean_pred - confidence_level * std_pred

    # Ensure true values are flattened as well
    true_values = np.array(true_values).flatten()

  
    fig_width = 16
    fig_height = 9
    plt.figure(figsize=(fig_width, fig_height))
    
    # Plot true values and mean predictions
    plt.plot(strain_x_rescaled[key], true_values, label="True Values", color="b")
    plt.plot(strain_x_rescaled[key], mean_pred, label="Predicted Mean", color="r", linestyle='--')
    
    # Plot confidence intervals as a shaded region
    plt.fill_between(strain_x_rescaled[key], lower_bound, upper_bound, color="r", alpha=0.3, label=f"Confidence Interval (95%)")

    # Create metrics text with proper alignment using monospace font
    metrics_text = (
        f"MAE:   {mae:.2f} \n"
        f"RMSE:   {rmse:.2f} \n"
        f"MAPE:   {mape:.2f}%"
    )
    
    # Position the text box in the upper right corner with better styling
    plt.annotate(metrics_text, xy=(0.95, 0.95), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.6", facecolor='white', alpha=0.8, 
                           edgecolor='gray', linewidth=1),
                 ha='right', va='top', fontsize=14, family='monospace')

    # Increase font sizes
    plt.xlabel("Cycles")
    plt.ylabel("RUL (Cycles)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Run inference on validation and test datasets
true_val, pred_val, val_mae, val_rmse, val_mape = inference(model, val_data, device, key=test_key)
true_test, pred_test, test_mse, test_rmse, test_mape = inference(model, test_data, device, key=test_key)

# Plot for validation data
plot_predictions(true_val, pred_val, val_mae, val_rmse, val_mape, title=f"FOD{int(test_key.split('f')[-1])+3} Cross Validation Fold", key=test_key)

# Plot for test data
#plot_predictions(true_test, pred_test, test_mse, test_rmse, test_mape, title="Test Data: True vs Predicted RUL", key=test_key)

In [ ]:
# Run MC dropout inference on validation and test datasets
true_val, mc_pred_val, val_mae, val_rmse, val_mape = mc_dropout_inference(model, val_data, device, num_samples=100, key=test_key)
#true_test, mc_pred_test, test_mse, test_rmse, test_mape = mc_dropout_inference(model, test_data, device, num_samples=100, key='df3')

# Plot MC predictions for validation data
plot_mc_predictions(true_val, mc_pred_val, val_mae, val_rmse, val_mape, title=f"FOD{int(test_key.split('f')[-1])+3} Cross Validation Fold - RUL Prediction ", target_indexes=target_indexes[test_key], key=test_key)